# Importing a library that is not in Colaboratory

To import a library that's not in Colaboratory by default, you can use `!pip install` or `!apt-get install`.

In [ ]:
from google.colab import drive
drive.mount('/content/gdrive')

Mounted at /content/gdrive


In [ ]:

# unzip into /content/stanford_dogs
!unzip -q "/content/gdrive/MyDrive/42028AUT2025/assessment/archive (7).zip" -d /content/stanford_dogs

In [ ]:
!pip install -q torch torchvision lxml

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 119.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 92.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 56.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 11.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 40.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 19.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 99.0 MB/s eta 0:00:00


In [ ]:
import os
import torch
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from torchvision.models.detection import fasterrcnn_resnet50_fpn, FasterRCNN_ResNet50_FPN_Weights
import xml.etree.ElementTree as ET
from PIL import Image

In [ ]:
class StanfordDogsDetection(Dataset):
    def __init__(self, root, transforms=None):
        # adjust to your unzip layout
        self.img_dir = os.path.join(root, "images/Images")
        self.ann_dir = os.path.join(root, "annotations/Annotation")
        self.transforms = transforms

        # collect (breed, basename) pairs
        self.examples = []
        for breed in os.listdir(self.img_dir):
            breed_img_dir = os.path.join(self.img_dir, breed)
            if not os.path.isdir(breed_img_dir):
                continue
            for fname in os.listdir(breed_img_dir):
                if fname.lower().endswith(".jpg"):
                    basename = fname[:-4]
                    # only include if an annotation file exists
                    if os.path.exists(os.path.join(self.ann_dir, breed, basename)):
                        self.examples.append((breed, basename))

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, idx):
        breed, basename = self.examples[idx]
        img_path = os.path.join(self.img_dir, breed, basename + ".jpg")
        ann_path = os.path.join(self.ann_dir, breed, basename)  # no ".xml"

        # load image
        img = Image.open(img_path).convert("RGB")

        # parse Pascal VOC–style XML even if file has no extension
        tree = ET.parse(ann_path)
        boxes, labels = [], []
        for obj in tree.findall(".//object"):
            bbox = obj.find("bndbox")
            boxes.append([
                float(bbox.find("xmin").text),
                float(bbox.find("ymin").text),
                float(bbox.find("xmax").text),
                float(bbox.find("ymax").text),
            ])
            labels.append(1)  # single class "dog"

        target = {
            "boxes": torch.tensor(boxes, dtype=torch.float32),
            "labels": torch.tensor(labels, dtype=torch.int64),
        }

        if self.transforms:
            img = self.transforms(img)

        return img, target

In [ ]:
transform = transforms.Compose([transforms.ToTensor()])
dataset = StanfordDogsDetection("/content/stanford_dogs", transforms=transform)

num_epochs  = 50
batch_size  = 4

# split train/test
train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size
train_ds, test_ds = torch.utils.data.random_split(dataset, [train_size, test_size])

def collate_fn(batch):
    return tuple(zip(*batch))

train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, collate_fn=collate_fn)
test_loader  = DataLoader(test_ds,  batch_size=batch_size, shuffle=False, collate_fn=collate_fn)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

weights = FasterRCNN_ResNet50_FPN_Weights.DEFAULT
model = fasterrcnn_resnet50_fpn(weights=weights).to(device)
model.train()

optimizer = optim.SGD(
    model.parameters(), lr=2e-3, momentum=0.9, weight_decay=1e-4
)

Downloading: "https://download.pytorch.org/models/fasterrcnn_resnet50_fpn_coco-258fb6c6.pth" to /root/.cache/torch/hub/checkpoints/fasterrcnn_resnet50_fpn_coco-258fb6c6.pth
100%|██████████| 160M/160M [00:00<00:00, 204MB/s]


In [ ]:
for epoch in range(num_epochs):
    epoch_loss = 0.0
    for images, targets in train_loader:
        images  = [img.to(device) for img in images]
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

        loss_dict = model(images, targets)
        losses    = sum(loss_dict.values())

        optimizer.zero_grad()
        losses.backward()
        optimizer.step()

        epoch_loss += losses.item()

    avg_loss = epoch_loss / len(train_loader)
    print(f"Epoch {epoch+1}/{num_epochs} — avg loss: {avg_loss:.4f}")

Epoch 1/50 — avg loss: 0.0689
Epoch 2/50 — avg loss: 0.0591
Epoch 3/50 — avg loss: 0.0547
Epoch 4/50 — avg loss: 0.0510
Epoch 5/50 — avg loss: 0.0480
Epoch 6/50 — avg loss: 0.0459
Epoch 7/50 — avg loss: 0.0440
Epoch 8/50 — avg loss: 0.0421
Epoch 9/50 — avg loss: 0.0406
Epoch 10/50 — avg loss: 0.0391
Epoch 11/50 — avg loss: 0.0376
Epoch 12/50 — avg loss: 0.0366
Epoch 13/50 — avg loss: 0.0361
Epoch 14/50 — avg loss: 0.0346


In [ ]:
# take 3 images from test set
images, targets = next(iter(test_loader))
images = [img.to(device) for img in images]
targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

# ▶︎ 2. Run inference
with torch.no_grad():
    preds = model(images)

# ▶︎ 3. Visualization helper
import torchvision
import matplotlib.pyplot as plt

def show_boxes(img_tensor, boxes, title="", ax=None, color="red"):
    # img_tensor = [C×H×W] in [0,1]
    img = torchvision.transforms.ToPILImage()(img_tensor.cpu())
    ax = ax or plt.gca()
    ax.imshow(img)
    for box in boxes.cpu().numpy():
        x1, y1, x2, y2 = box
        rect = plt.Rectangle((x1,y1), x2-x1, y2-y1,
                             fill=False, edgecolor=color, linewidth=2)
        ax.add_patch(rect)
    ax.set_xticks([]); ax.set_yticks([])
    ax.set_title(title)

# ▶︎ 4. Plot side-by-side
fig, axes = plt.subplots(3, 2, figsize=(10,15))
for i in range(3):
    # ground-truth on left
    gt_boxes = targets[i]["boxes"]
    show_boxes(images[i], gt_boxes, title="GT", ax=axes[i,0], color="green")
    # predictions on right (only high-conf detections)
    pred_boxes = preds[i]["boxes"][preds[i]["scores"]>0.5]
    show_boxes(images[i], pred_boxes, title="Pred (score>0.5)", ax=axes[i,1], color="red")

plt.tight_layout()
plt.show()

# Install [cartopy](http://scitools.org.uk/cartopy/docs/latest/)